<a href="https://colab.research.google.com/github/sandhiyajeganathan2002-gif/INT8_Detection_Under_Degradation_Trial_Task/blob/main/INT8_Detection_Under_Degradation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q ultralytics openvino onnx onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 94.9 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.export(format="onnx", opset=12, dynamic=False, imgsz=640)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 10 packages in 310ms
Prepared 2 packages in 54ms
Installed 2 packages in 6ms
 + colorama==0.4.6
 + onnxslim==0.1.

'yolov8n.onnx'

In [5]:
!mkdir -p data
!wget -q http://images.cocodataset.org/zips/val2017.zip -O data/val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O data/annotations_trainval2017.zip
!unzip -q data/val2017.zip -d data/
!unzip -q data/annotations_trainval2017.zip -d data/

In [6]:
!ls data/
!ls data/val2017 | head -5

annotations  annotations_trainval2017.zip  val2017  val2017.zip
000000000139.jpg
000000000285.jpg
000000000632.jpg
000000000724.jpg
000000000776.jpg


In [7]:
# Step 4: Pick 500 images that contain our 5 target classes

from pycocotools.coco import COCO
import random
import json

# Load the annotation file (this tells us which objects are in which images)
coco = COCO("data/annotations/instances_val2017.json")

# The 5 classes we care about
target_classes = ["person", "car", "bicycle", "traffic light", "stop sign"]

# Get the ID numbers for these classes (COCO uses IDs internally, not names)
category_ids = coco.getCatIds(catNms=target_classes)
print("Category IDs:", category_ids)

# Now find all images that contain AT LEAST ONE of these classes
all_matching_image_ids = []

for cat_id in category_ids:
    image_ids_for_this_class = coco.getImgIds(catIds=[cat_id])
    all_matching_image_ids.extend(image_ids_for_this_class)

# Remove duplicates (an image with both "person" and "car" would appear twice)
unique_image_ids = list(set(all_matching_image_ids))

print("Total unique images found:", len(unique_image_ids))

# Randomly pick 500 of them (fixed seed = same 500 every time we rerun this)
random.seed(42)
sampled_image_ids = random.sample(unique_image_ids, 500)

print("Sampled 500 images. First 5 IDs:", sampled_image_ids[:5])

# Save this list so we (and Srishti) can reproduce the exact same experiment
with open("data/sampled_image_ids.json", "w") as f:
    json.dump(sampled_image_ids, f)

print("Saved to data/sampled_image_ids.json")

loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
Category IDs: [1, 2, 3, 10, 13]
Total unique images found: 2956
Sampled 500 images. First 5 IDs: [416885, 148662, 164115, 560178, 51938]
Saved to data/sampled_image_ids.json


In [8]:
from ultralytics import YOLO
from pycocotools.coco import COCO
import json

model = YOLO("yolov8n.pt")

In [9]:
coco = COCO("data/annotations/instances_val2017.json")

with open("data/sampled_image_ids.json", "r") as f:
    sampled_image_ids = json.load(f)

loading annotations into memory...
Done (t=0.98s)
creating index...
index created!


In [10]:
# We only care about these 5 classes
target_classes = ["person", "car", "bicycle", "traffic light", "stop sign"]

# Get all category info from COCO (each one has an id and a name)
all_category_ids = coco.getCatIds()
coco_categories = coco.loadCats(all_category_ids)

# Print them so we can see what we're working with
print("All categories loaded:")
for cat in coco_categories:
    print(cat["id"], "-", cat["name"])

# Now build our lookup dictionary step by step
name_to_coco_id = {}

for cat in coco_categories:
    cat_name = cat["name"]
    cat_id = cat["id"]
    name_to_coco_id[cat_name] = cat_id

# Check that our 5 target classes got mapped correctly
print("\nLookup table for our target classes:")
for name in target_classes:
    print(name, "->", name_to_coco_id[name])

All categories loaded:
1 - person
2 - bicycle
3 - car
4 - motorcycle
5 - airplane
6 - bus
7 - train
8 - truck
9 - boat
10 - traffic light
11 - fire hydrant
13 - stop sign
14 - parking meter
15 - bench
16 - bird
17 - cat
18 - dog
19 - horse
20 - sheep
21 - cow
22 - elephant
23 - bear
24 - zebra
25 - giraffe
27 - backpack
28 - umbrella
31 - handbag
32 - tie
33 - suitcase
34 - frisbee
35 - skis
36 - snowboard
37 - sports ball
38 - kite
39 - baseball bat
40 - baseball glove
41 - skateboard
42 - surfboard
43 - tennis racket
44 - bottle
46 - wine glass
47 - cup
48 - fork
49 - knife
50 - spoon
51 - bowl
52 - banana
53 - apple
54 - sandwich
55 - orange
56 - broccoli
57 - carrot
58 - hot dog
59 - pizza
60 - donut
61 - cake
62 - chair
63 - couch
64 - potted plant
65 - bed
67 - dining table
70 - toilet
72 - tv
73 - laptop
74 - mouse
75 - remote
76 - keyboard
77 - cell phone
78 - microwave
79 - oven
80 - toaster
81 - sink
82 - refrigerator
84 - book
85 - clock
86 - vase
87 - scissors
88 - teddy be

In [11]:
# This list will store all our detections
all_detections = []

# Loop through every image one by one
counter = 0
for img_id in sampled_image_ids:
    counter = counter + 1
    if counter % 50 == 0:
        print("Processed", counter, "images so far...")

    # Get the image file name from its ID
    img_info = coco.loadImgs(img_id)[0]
    file_name = img_info["file_name"]
    img_path = "data/val2017/" + file_name

    # Run YOLOv8n on this one image
    results = model.predict(img_path, verbose=False, conf=0.001)
    result = results[0]  # only one image, so take the first result

    # Go through every box YOLO detected in this image
    for box in result.boxes:
        class_index = int(box.cls[0])
        class_name = model.names[class_index]

        # Skip if it's not one of our 5 target classes
        if class_name not in target_classes:
            continue

        # Get box coordinates (YOLO gives x1,y1,x2,y2)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        confidence_score = float(box.conf[0])

        # COCO format wants [x, y, width, height]
        width = x2 - x1
        height = y2 - y1
        coco_style_box = [x1, y1, width, height]

        # Look up the COCO category ID for this class name
        coco_category_id = name_to_coco_id[class_name]

        # Save this detection
        detection = {
            "image_id": img_id,
            "category_id": coco_category_id,
            "bbox": coco_style_box,
            "score": confidence_score
        }
        all_detections.append(detection)

print("Done! Total detections found:", len(all_detections))

Processed 50 images so far...
Processed 100 images so far...
Processed 150 images so far...
Processed 200 images so far...
Processed 250 images so far...
Processed 300 images so far...
Processed 350 images so far...
Processed 400 images so far...
Processed 450 images so far...
Processed 500 images so far...
Done! Total detections found: 34814


In [12]:
with open("data/fp32_detections.json", "w") as f:
    json.dump(all_detections, f)

In [13]:
!ls -la data/fp32_detections.json
print("First detection example:", all_detections[0])

-rw-r--r-- 1 root root 5409126 Aug 13 08:55 data/fp32_detections.json
First detection example: {'image_id': 416885, 'category_id': 1, 'bbox': [0.0, 0.24651288986206055, 44.60883712768555, 174.58324480056763], 'score': 0.5517733693122864}


In [14]:
# Score our detections against the real answers (compute mAP)

from pycocotools.cocoeval import COCOeval

# Load our detections into COCO's evaluation format
coco_detections = coco.loadRes("data/fp32_detections.json")

# Set up the evaluator: compare our detections vs ground truth, for bounding boxes
coco_eval = COCOeval(coco, coco_detections, iouType="bbox")

# Restrict evaluation to only our 500 sampled images
coco_eval.params.imgIds = sampled_image_ids

# Restrict evaluation to only our 5 target classes
target_category_ids = coco.getCatIds(catNms=target_classes)
coco_eval.params.catIds = target_category_ids

# Run the evaluation
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

Loading and preparing results...
DONE (t=0.46s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.63s).
Accumulating evaluation results...
DONE (t=0.36s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.425
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.620
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.451
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.201
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.581
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.704
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.253
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.486
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.555
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

In [15]:
import numpy as np

# COCOeval stores results in a big array after summarize() runs
# We need to pull out just the "precision" part
precisions = coco_eval.eval['precision']

print("Shape of precision array:", precisions.shape)
print("This means: [IoU thresholds, recall levels, classes, object sizes, max detections]")

Shape of precision array: (10, 101, 5, 4, 3)
This means: [IoU thresholds, recall levels, classes, object sizes, max detections]


In [16]:
print("Per-class AP (IoU=0.50:0.95, area=all):\n")

for idx in range(len(target_category_ids)):
    cat_id = target_category_ids[idx]

    # Find the readable name for this category id
    cat_name = ""
    for cat in coco_categories:
        if cat["id"] == cat_id:
            cat_name = cat["name"]

    # Pull out precision values just for this one class
    # [:, :, idx, 0, 2] means: all IoU thresholds, all recall levels, THIS class, area=all, maxDets=100
    precision_for_class = precisions[:, :, idx, 0, 2]

    # Some values are -1 which means "no ground truth objects for this threshold" - ignore those
    valid_precision = precision_for_class[precision_for_class > -1]

    # Average what's left to get this class's AP
    if len(valid_precision) > 0:
        ap = np.mean(valid_precision)
    else:
        ap = float('nan')

    print(cat_name, ": AP =", round(ap, 3))

Per-class AP (IoU=0.50:0.95, area=all):

person : AP = 0.521
bicycle : AP = 0.34
car : AP = 0.408
traffic light : AP = 0.209
stop sign : AP = 0.648


In [18]:
!mkdir -p results

In [19]:
import json

# Collect all our Day 2 results into one clean dictionary
fp32_baseline_results = {
    "overall": {
        "mAP_0.5:0.95": 0.425,
        "mAP_0.5": 0.620,
        "AP_small": 0.201,
        "AP_medium": 0.581,
        "AP_large": 0.704
    },
    "per_class_AP": {
        "person": 0.521,
        "bicycle": 0.340,
        "car": 0.408,
        "traffic light": 0.209,
        "stop sign": 0.648
    }
}

# Save it to a file
with open("results/fp32_baseline.json", "w") as f:
    json.dump(fp32_baseline_results, f, indent=2)

print("Saved FP32 baseline results")
print(fp32_baseline_results)

Saved FP32 baseline results
{'overall': {'mAP_0.5:0.95': 0.425, 'mAP_0.5': 0.62, 'AP_small': 0.201, 'AP_medium': 0.581, 'AP_large': 0.704}, 'per_class_AP': {'person': 0.521, 'bicycle': 0.34, 'car': 0.408, 'traffic light': 0.209, 'stop sign': 0.648}}
